# Validation Data Audit

Prepare and audit the March–April 2025 CFPB archive for chronological validation.

Verify the source schema, complaint IDs, received dates, and credit card narrative coverage before evaluating any model.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import zipfile

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
assert (PROJECT_ROOT / "PROJECT_PLAN.md").is_file(), (
    "Run this notebook from the project's notebooks directory."
)

SOURCE_ID = "cfpb_2025_mar_apr"
ARCHIVE_NAME = "CCDB_Export_10_March_2025_through_April_2025.zip"
DOWNLOAD_URL = f"https://files.consumerfinance.gov/f/documents/{ARCHIVE_NAME}"

zip_path = PROJECT_ROOT / "data" / "raw" / ARCHIVE_NAME
manifest_path = PROJECT_ROOT / "data" / "manifests" / f"{SOURCE_ID}.json"

assert zip_path.is_file(), f"Archive not found: {zip_path}"

In [2]:
# Compute the archive fingerprint in chunks.
sha256 = hashlib.sha256()

with zip_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

with zipfile.ZipFile(zip_path) as archive:
    csv_members = [
        member
        for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".csv")
    ]

    if len(csv_members) != 1:
        raise ValueError("Expected exactly one CSV file in the archive.")

    member = csv_members[0]
    csv_path = zip_path.parent / Path(member.filename).name
    temp_path = csv_path.with_suffix(".csv.part")

    # Publish the CSV only after extraction and size verification succeed.
    with archive.open(member) as source, temp_path.open("wb") as destination:
        shutil.copyfileobj(source, destination, length=1024 * 1024)

    if temp_path.stat().st_size != member.file_size:
        raise ValueError("Extracted file size does not match archive metadata.")

    temp_path.replace(csv_path)

manifest = {
    "dataset": "CFPB Consumer Complaint Narratives Archive",
    "source_id": SOURCE_ID,
    "source_label": "March 2025 through April 2025",
    "download_url": DOWNLOAD_URL,
    "local_file": zip_path.relative_to(PROJECT_ROOT).as_posix(),
    "size_bytes": zip_path.stat().st_size,
    "sha256": sha256.hexdigest(),
    "archive_members": [
        {
            "name": member.filename,
            "uncompressed_size_bytes": member.file_size,
        }
    ],
    "manifest_created_at_utc": datetime.now(timezone.utc).isoformat(),
    "date_coverage_verified": False,
}

manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("CSV:", csv_path.relative_to(PROJECT_ROOT).as_posix())
print("CSV size (bytes):", csv_path.stat().st_size)
print("Manifest:", manifest_path.relative_to(PROJECT_ROOT).as_posix())

CSV: data/raw/CCDB_Export_10_March_2025_through_April_2025.csv
CSV size (bytes): 452458891
Manifest: data/manifests/cfpb_2025_mar_apr.json


In [3]:
training_csv_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "CCDB_Export_9_January_2025_through_February_2025.csv"
)

training_columns = pd.read_csv(training_csv_path, nrows=0).columns.tolist()
validation_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()

same_schema = validation_columns == training_columns

print("Same column names and order:", same_schema)
print(
    "Columns missing from validation:",
    sorted(set(training_columns) - set(validation_columns)),
)
print(
    "Additional columns in validation:",
    sorted(set(validation_columns) - set(training_columns)),
)

assert same_schema, "Validation schema differs from the training source."

con = duckdb.connect()

con.read_csv(
    str(csv_path),
    header=True,
    all_varchar=True,
).create_view("complaints_raw", replace=True)

source_audit_sql = (
    PROJECT_ROOT / "sql" / "01_initial_audit.sql"
).read_text(encoding="utf-8")

product_summary_sql = (
    PROJECT_ROOT / "sql" / "02_product_summary.sql"
).read_text(encoding="utf-8")

validation_source_audit = con.execute(source_audit_sql).df()
validation_product_summary = con.execute(product_summary_sql).df()

print("\nValidation source audit:")
display(validation_source_audit.T.rename(columns={0: "value"}))

print("\nCredit card subset:")
display(
    validation_product_summary.loc[
        validation_product_summary["product"].eq("Credit card")
    ]
)

Same column names and order: True
Columns missing from validation: []
Additional columns in validation: []

Validation source audit:


,value
total_rows,803073
distinct_complaint_ids,803073
missing_complaint_ids,0
earliest_received_date,2025-03-01 00:00:00
latest_received_date,2025-04-30 00:00:00
missing_received_dates,0
unparsed_received_dates,0
rows_with_narrative,210147
narrative_coverage_pct,26.17



Credit card subset:


,product,total_rows,rows_with_narrative,narrative_coverage_pct
2,Credit card,13152,5957,45.29


In [4]:
audit = validation_source_audit.iloc[0]

assert int(audit["total_rows"]) > 0, "The validation source is empty."
assert int(audit["missing_complaint_ids"]) == 0, (
    "Missing complaint IDs found."
)
assert int(audit["total_rows"]) == int(audit["distinct_complaint_ids"]), (
    "Duplicate complaint IDs found within the validation source."
)
assert int(audit["missing_received_dates"]) == 0, (
    "Missing received dates found."
)
assert int(audit["unparsed_received_dates"]) == 0, (
    "Unparsed received dates found."
)

earliest_date = pd.Timestamp(audit["earliest_received_date"])
latest_date = pd.Timestamp(audit["latest_received_date"])

assert earliest_date == pd.Timestamp("2025-03-01"), (
    "Unexpected earliest received date."
)
assert latest_date == pd.Timestamp("2025-04-30"), (
    "Unexpected latest received date."
)

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

manifest["date_coverage_verified"] = True
manifest["observed_date_coverage"] = {
    "earliest_received_date": earliest_date.date().isoformat(),
    "latest_received_date": latest_date.date().isoformat(),
    "missing_received_dates": int(audit["missing_received_dates"]),
    "unparsed_received_dates": int(audit["unparsed_received_dates"]),
}
manifest["observed_counts"] = {
    "total_rows": int(audit["total_rows"]),
    "distinct_complaint_ids": int(audit["distinct_complaint_ids"]),
    "rows_with_narrative": int(audit["rows_with_narrative"]),
}

manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Validation schema, ID, and date checks passed.")
print("Manifest date coverage verified.")

Validation schema, ID, and date checks passed.
Manifest date coverage verified.


In [5]:
training_archive_names = [
    "CCDB_Export_6_April_2024_through_July_2024.csv",
    "CCDB_Export_7_August_2024_through_October_2024.csv",
    "CCDB_Export_8_November_2024_through_December_2024.csv",
    "CCDB_Export_9_January_2025_through_February_2025.csv",
]

training_csv_paths = [
    PROJECT_ROOT / "data" / "raw" / name
    for name in training_archive_names
]

for path in training_csv_paths:
    assert path.is_file(), f"Training source not found: {path}"

con.read_csv(
    [str(path) for path in training_csv_paths],
    header=True,
    all_varchar=True,
).create_view("training_sources_raw", replace=True)

validation_id_overlap_sql = """
WITH training_ids AS (
    SELECT DISTINCT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id
    FROM training_sources_raw
),
validation_ids AS (
    SELECT DISTINCT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id
    FROM complaints_raw
)
SELECT
    COUNT(*) AS overlapping_complaint_ids
FROM validation_ids AS v
JOIN training_ids AS t
    ON v.complaint_id = t.complaint_id
"""

validation_id_overlap = con.execute(validation_id_overlap_sql).df()
display(validation_id_overlap)

assert int(
    validation_id_overlap.loc[0, "overlapping_complaint_ids"]
) == 0, "Complaint IDs overlap between training and validation."

print("Training-validation complaint ID overlap check passed.")

,overlapping_complaint_ids
0,0


Training-validation complaint ID overlap check passed.


In [6]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

validation_source_audit.assign(source_id=SOURCE_ID).to_csv(
    reports_dir / "validation_source_audit.csv",
    index=False,
)

validation_product_summary.to_csv(
    reports_dir / "validation_product_summary.csv",
    index=False,
)

validation_id_overlap.to_csv(
    reports_dir / "validation_id_overlap.csv",
    index=False,
)

(PROJECT_ROOT / "sql" / "validation_id_overlap.sql").write_text(
    validation_id_overlap_sql.strip() + "\n",
    encoding="utf-8",
)

print("Validation source audit, product summary, and ID overlap report saved.")

Validation source audit, product summary, and ID overlap report saved.


In [7]:
validation_credit_card_extraction_sql = """
SELECT
    NULLIF(TRIM("Complaint ID"), '') AS complaint_id,
    CAST(
        TRY_STRPTIME(
            TRIM("Date received"),
            ['%Y-%m-%d', '%m/%d/%Y']
        ) AS DATE
    ) AS received_date,
    TRIM("Product") AS product,
    NULLIF(TRIM("Sub-product"), '') AS sub_product,
    NULLIF(TRIM("Issue"), '') AS issue,
    NULLIF(TRIM("Sub-issue"), '') AS sub_issue,
    NULLIF(TRIM("Consumer complaint narrative"), '') AS narrative,
    NULLIF(TRIM("Company"), '') AS company,
    NULLIF(TRIM("Submitted via"), '') AS submitted_via
FROM complaints_raw
WHERE TRIM("Product") = 'Credit card'
ORDER BY received_date, complaint_id
"""

credit_card_validation_all = con.execute(
    validation_credit_card_extraction_sql
).df()

credit_card_validation_all["source_file"] = csv_path.name

credit_card_validation_text = credit_card_validation_all.loc[
    credit_card_validation_all["narrative"].notna()
].copy().reset_index(drop=True)

credit_card_counts = validation_product_summary.loc[
    validation_product_summary["product"].eq("Credit card")
].iloc[0]

assert len(credit_card_validation_all) == int(
    credit_card_counts["total_rows"]
), "Credit card extraction count mismatch."

assert len(credit_card_validation_text) == int(
    credit_card_counts["rows_with_narrative"]
), "Credit card narrative count mismatch."

for frame in [credit_card_validation_all, credit_card_validation_text]:
    assert frame["complaint_id"].notna().all(), "Missing complaint IDs."
    assert frame["complaint_id"].is_unique, "Duplicate complaint IDs."
    assert frame["received_date"].notna().all(), "Missing received dates."
    assert frame["received_date"].between(
        pd.Timestamp("2025-03-01"),
        pd.Timestamp("2025-04-30"),
    ).all(), "Received dates outside the validation period."
    assert frame["issue"].notna().all(), "Missing issue labels."

assert credit_card_validation_text["narrative"].str.strip().ne("").all(), (
    "Empty narratives found in the text subset."
)

validation_extraction_summary = pd.DataFrame([
    {
        "dataset": "credit_card_validation_all",
        "rows": len(credit_card_validation_all),
        "missing_issue_rows": int(
            credit_card_validation_all["issue"].isna().sum()
        ),
    },
    {
        "dataset": "credit_card_validation_text",
        "rows": len(credit_card_validation_text),
        "missing_issue_rows": int(
            credit_card_validation_text["issue"].isna().sum()
        ),
    },
])

display(validation_extraction_summary)
print("Validation extraction count, ID, label, and date checks passed.")

,dataset,rows,missing_issue_rows
0,credit_card_validation_all,13152,0
1,credit_card_validation_text,5957,0


Validation extraction count, ID, label, and date checks passed.


In [8]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

validation_outputs = [
    ("credit_card_validation_all", credit_card_validation_all),
    ("credit_card_validation_text", credit_card_validation_text),
]

for name, frame in validation_outputs:
    output_path = processed_dir / f"{name}.parquet"
    frame.to_parquet(output_path, index=False)

    restored = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(restored, frame)

    print("Saved:", output_path.relative_to(PROJECT_ROOT).as_posix())
    print("Rows verified:", len(restored))
    print("Size (MB):", round(output_path.stat().st_size / 1024**2, 2))
    print("-" * 60)

validation_extraction_summary.to_csv(
    reports_dir / "validation_extraction_summary.csv",
    index=False,
)

(PROJECT_ROOT / "sql" / "validation_credit_card_extraction.sql").write_text(
    validation_credit_card_extraction_sql.strip() + "\n",
    encoding="utf-8",
)

Saved: data/processed/credit_card_validation_all.parquet
Rows verified: 13152
Size (MB): 3.73
------------------------------------------------------------
Saved: data/processed/credit_card_validation_text.parquet
Rows verified: 5957
Size (MB): 3.67
------------------------------------------------------------


640

In [9]:
baseline_path = processed_dir / "baseline_training.parquet"
training_text_path = processed_dir / "credit_card_training_text.parquet"

assert baseline_path.is_file(), "Baseline training sample not found."
assert training_text_path.is_file(), "Training text dataset not found."

con.read_parquet(str(baseline_path)).create_view(
    "baseline_training_reference", replace=True
)
con.read_parquet(str(training_text_path)).create_view(
    "all_training_text_reference", replace=True
)
con.register("validation_text_input", credit_card_validation_text)

validation_text_overlap_sql = r"""
WITH validation_normalized AS (
    SELECT
        *,
        LOWER(
            REGEXP_REPLACE(TRIM(narrative), '\s+', ' ', 'g')
        ) AS text_key
    FROM validation_text_input
),
baseline_keys AS (
    SELECT DISTINCT
        LOWER(
            REGEXP_REPLACE(TRIM(narrative), '\s+', ' ', 'g')
        ) AS text_key
    FROM baseline_training_reference
),
all_training_keys AS (
    SELECT DISTINCT
        LOWER(
            REGEXP_REPLACE(TRIM(narrative), '\s+', ' ', 'g')
        ) AS text_key
    FROM all_training_text_reference
)
SELECT
    v.*,
    MD5(v.text_key) AS text_group_id,
    b.text_key IS NOT NULL AS exact_overlap_with_baseline,
    t.text_key IS NOT NULL AS exact_overlap_with_training_period
FROM validation_normalized AS v
LEFT JOIN baseline_keys AS b
    ON v.text_key = b.text_key
LEFT JOIN all_training_keys AS t
    ON v.text_key = t.text_key
ORDER BY v.received_date, v.complaint_id
"""

validation_text_flagged = con.execute(validation_text_overlap_sql).df()

assert len(validation_text_flagged) == len(credit_card_validation_text)
assert validation_text_flagged["complaint_id"].is_unique
assert not (
    validation_text_flagged["exact_overlap_with_baseline"]
    & ~validation_text_flagged["exact_overlap_with_training_period"]
).any(), "Baseline overlap must also be training-period overlap."

overlap_summary_rows = []

for reference, flag_column in [
    ("baseline_training", "exact_overlap_with_baseline"),
    ("all_training_period_text", "exact_overlap_with_training_period"),
]:
    overlap_count = int(validation_text_flagged[flag_column].sum())

    overlap_summary_rows.append({
        "reference": reference,
        "validation_text_rows": len(validation_text_flagged),
        "exact_overlap_rows": overlap_count,
        "no_exact_overlap_rows": len(validation_text_flagged) - overlap_count,
        "exact_overlap_pct": round(
            100 * overlap_count / len(validation_text_flagged), 2
        ),
    })

validation_text_overlap_summary = pd.DataFrame(overlap_summary_rows)
display(validation_text_overlap_summary)

print("Validation text overlap flags created; all validation rows retained.")

,reference,validation_text_rows,exact_overlap_rows,no_exact_overlap_rows,exact_overlap_pct
0,baseline_training,5957,121,5836,2.03
1,all_training_period_text,5957,164,5793,2.75


Validation text overlap flags created; all validation rows retained.


In [10]:
# Add quality flags without removing validation records.
validation_group_stats = (
    validation_text_flagged
    .groupby("text_key", sort=False)
    .agg(
        validation_text_group_size=("complaint_id", "size"),
        validation_distinct_issue_labels=("issue", "nunique"),
    )
    .reset_index()
)

validation_text_flagged = validation_text_flagged.merge(
    validation_group_stats,
    on="text_key",
    how="left",
    validate="many_to_one",
)

validation_text_flagged["is_repeated_within_validation"] = (
    validation_text_flagged["validation_text_group_size"].gt(1)
)

validation_text_flagged["has_validation_label_conflict"] = (
    validation_text_flagged["validation_distinct_issue_labels"].gt(1)
)

baseline_labels = set(
    con.execute(
        "SELECT DISTINCT issue FROM baseline_training_reference"
    ).df()["issue"]
)

validation_text_flagged["label_seen_in_baseline"] = (
    validation_text_flagged["issue"].isin(baseline_labels)
)

assert len(validation_text_flagged) == 5957
assert validation_text_flagged["complaint_id"].is_unique

validation_quality_summary = pd.DataFrame([{
    "total_text_rows": len(validation_text_flagged),
    "distinct_text_keys": len(validation_group_stats),
    "repeated_text_groups": int(
        validation_group_stats["validation_text_group_size"].gt(1).sum()
    ),
    "rows_in_repeated_groups": int(
        validation_text_flagged["is_repeated_within_validation"].sum()
    ),
    "label_conflict_groups": int(
        validation_group_stats["validation_distinct_issue_labels"].gt(1).sum()
    ),
    "label_conflict_rows": int(
        validation_text_flagged["has_validation_label_conflict"].sum()
    ),
    "unseen_label_rows": int(
        (~validation_text_flagged["label_seen_in_baseline"]).sum()
    ),
}])

display(validation_quality_summary.T.rename(columns={0: "value"}))

unseen_labels = sorted(
    set(validation_text_flagged["issue"]) - baseline_labels
)
print("Labels absent from baseline training:", unseen_labels)

,value
total_text_rows,5957
distinct_text_keys,5746
repeated_text_groups,29
rows_in_repeated_groups,240
label_conflict_groups,1
label_conflict_rows,3
unseen_label_rows,0


Labels absent from baseline training: []


In [11]:
validation_class_summary = (
    validation_text_flagged
    .groupby("issue")
    .agg(
        validation_text_rows=("complaint_id", "size"),
        exact_overlap_with_training_period_rows=(
            "exact_overlap_with_training_period", "sum"
        ),
    )
    .reset_index()
)

validation_class_summary["no_training_exact_overlap_rows"] = (
    validation_class_summary["validation_text_rows"]
    - validation_class_summary["exact_overlap_with_training_period_rows"]
)

validation_class_summary["label_seen_in_baseline"] = (
    validation_class_summary["issue"].isin(baseline_labels)
)

validation_class_summary = validation_class_summary.sort_values(
    ["validation_text_rows", "issue"],
    ascending=[False, True],
).reset_index(drop=True)

assert validation_class_summary["validation_text_rows"].sum() == 5957
assert validation_class_summary["no_training_exact_overlap_rows"].sum() == 5793

display(validation_class_summary)

,issue,validation_text_rows,exact_overlap_with_training_period_rows,no_training_exact_overlap_rows,label_seen_in_baseline
0,Problem with a purchase shown on your statement,1605,0,1605,True
1,Getting a credit card,797,110,687,True
2,"Other features, terms, or problems",744,0,744,True
3,Fees or interest,701,0,701,True
4,Closing your account,552,0,552,True
5,Problem when making payments,387,0,387,True
6,"Advertising and marketing, including promotion...",288,0,288,True
7,Trouble using your card,258,0,258,True
8,Incorrect information on your report,255,4,251,True
9,Problem with a company's investigation into an...,223,50,173,True


In [12]:
validation_text_flagged["is_no_training_exact_overlap"] = (
    ~validation_text_flagged["exact_overlap_with_training_period"]
)

flagged_path = processed_dir / "credit_card_validation_text_flagged.parquet"
validation_text_flagged.to_parquet(flagged_path, index=False)

pd.testing.assert_frame_equal(
    pd.read_parquet(flagged_path),
    validation_text_flagged,
)

for name, frame in [
    ("validation_text_overlap_summary", validation_text_overlap_summary),
    ("validation_text_quality_summary", validation_quality_summary),
    ("validation_class_summary", validation_class_summary),
]:
    frame.to_csv(reports_dir / f"{name}.csv", index=False)

_ = (PROJECT_ROOT / "sql" / "validation_text_overlap.sql").write_text(
    validation_text_overlap_sql.strip() + "\n",
    encoding="utf-8",
)

print("Flagged validation data saved and verified.")
print("Full validation rows:", len(validation_text_flagged))
print(
    "No training-period exact overlap rows:",
    int(validation_text_flagged["is_no_training_exact_overlap"].sum()),
)

Flagged validation data saved and verified.
Full validation rows: 5957
No training-period exact overlap rows: 5793


In [13]:
def validation_file_metadata(path):
    """Record a file's relative path, size, and SHA-256 fingerprint."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return {
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": digest.hexdigest(),
    }


validation_config = {
    "policy_version": 1,
    "source_id": SOURCE_ID,
    "date_start": "2025-03-01",
    "date_end": "2025-04-30",
    "model_input": "narrative",
    "target": "issue",
    "full_validation_rows": len(validation_text_flagged),
    "comparison_subset": {
        "flag_column": "is_no_training_exact_overlap",
        "required_value": True,
        "rows": int(
            validation_text_flagged["is_no_training_exact_overlap"].sum()
        ),
        "reference": "All training-period credit card narratives",
    },
    "text_normalization": {
        "trim_outer_whitespace": True,
        "lowercase": True,
        "collapse_whitespace": True,
        "retain_punctuation_numbers_and_redactions": True,
    },
    "remove_validation_exact_duplicates": False,
    "remove_validation_label_conflicts": False,
    "automatic_near_duplicate_removal": False,
    "fit_preprocessing_on_validation": False,
    "baseline_labels": sorted(baseline_labels),
    "unseen_validation_labels": unseen_labels,
    "reference_files": [
        validation_file_metadata(baseline_path),
        validation_file_metadata(training_text_path),
    ],
    "validation_file": validation_file_metadata(flagged_path),
}

config_path = PROJECT_ROOT / "configs" / "validation_data.json"
config_path.parent.mkdir(parents=True, exist_ok=True)

_ = config_path.write_text(
    json.dumps(validation_config, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Validation configuration saved.")

Validation configuration saved.


In [14]:
quality = validation_quality_summary.iloc[0]

validation_report = f"""# Validation Data Audit

## Source and integrity

- Source: March–April 2025 CFPB Consumer Complaint Narratives Archive.
- Received-date range: 2025-03-01 through 2025-04-30.
- Source rows: {int(audit["total_rows"]):,}.
- Column names and order match the training source.
- Complaint IDs are unique and non-missing.
- Received dates are valid and non-missing.
- Complaint ID overlap with all four training sources: 0.
- Credit card complaints: {len(credit_card_validation_all):,}.
- Credit card complaints with narratives: {len(validation_text_flagged):,}.
- Missing issue labels in the text subset: 0.

## Text quality

- Distinct normalized texts: {int(quality["distinct_text_keys"]):,}.
- Repeated text groups: {int(quality["repeated_text_groups"]):,}.
- Rows in repeated groups: {int(quality["rows_in_repeated_groups"]):,}.
- Conflicting-label groups: {int(quality["label_conflict_groups"]):,}.
- Rows in conflicting-label groups: {int(quality["label_conflict_rows"]):,}.
- Rows with labels absent from baseline training: {int(quality["unseen_label_rows"]):,}.

## Evaluation populations

The full validation population retains all 5,957 credit card narratives,
including repeated texts and conflicting labels.

There are 121 rows with exact normalized text overlap with the baseline
training sample and 164 with the complete training-period text dataset.

A fixed secondary evaluation population contains 5,793 rows without exact
normalized text overlap with any training-period narrative. This same
population will be used when comparing training sample policies.

Normalization trims outer whitespace, converts text to lowercase, and
collapses consecutive whitespace. Punctuation, numbers, and redactions
are retained. Near duplicates and shared templates may still remain.

## Modeling rules and limitations

- Use narrative as the model input and issue as the target.
- Fit learned preprocessing on training data only.
- Retain original validation labels and quality flags.
- Keep calibration fitting separate from threshold evaluation.
- Do not use final-test outcomes for model selection.
- Rare validation classes have small sample counts; interpret their metrics
  alongside support counts.
- Complaint receipt dates do not establish historical public availability.
- This audit prepares evaluation data and does not report model performance.
"""

_ = (reports_dir / "validation_data_audit.md").write_text(
    validation_report,
    encoding="utf-8",
)

print("Validation audit report saved.")

Validation audit report saved.
